# FirstSlip Phase 2: open models (Qwen) with vLLM on Colab

Runs the **same frozen prompt** as the API models (`core/prompts.py`) on Qwen3.5-2B and Qwen3.5-4B. The output is run logs in the shared record format, which are scored locally by `scripts/phase2_score.py`.

- Runtime: **Colab Pro, A100**. Greedy decoding (temperature 0), thinking **off** (`enable_thinking=False`), so outputs stay in the two-line format.
- Each model is pinned to its Hugging Face commit SHA, and the SHA, `vllm` version and GPU are written into every run-log header.
- Run the cells top to bottom. **Step 1** (dev) is for prompt freezing. Run **Step 2** (test grid) only after the prompt is frozen and committed.
- At the end, download `phase2_runs.zip` and unzip it into the repo's `runs/` folder.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone https://github.com/AbdulRehmanAntall/firstslip.git
%cd firstslip
!git log -1 --format='%H %s'

In [ ]:
# vLLM brings its own torch; the repo's pinned scientific stack covers the rest.
!pip install -q vllm
!pip install -q -r requirements.txt
import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__)

Regenerate the synthetic data (deterministic, seed `20261019`) and check it against the committed hashes.

In [ ]:
!python scripts/generate_synthetic.py > /dev/null
import hashlib, json, pathlib
man = json.loads(pathlib.Path('data/synthetic/manifest.json').read_text())
for split, meta in man['splits'].items():
    h = hashlib.sha256(pathlib.Path(f'data/synthetic/{split}.jsonl').read_bytes()).hexdigest()
    assert h == meta['sha256'], f'{split}: hash mismatch'
    print(split, 'OK')

In [ ]:
import json, sys, subprocess, pathlib
sys.path.insert(0, '.')
from huggingface_hub import model_info
from vllm import LLM, SamplingParams
from core.phase2 import record, requests_for, run_config
from core.runlog import RunLog

CFG = json.loads(pathlib.Path('config/phase2.json').read_text())['open_models']
GPU = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).strip()
SAMPLING = dict(temperature=0.0, max_tokens=256)

def load(name, use_fallback=False):
    repo = CFG[name]['fallback' if use_fallback else 'model']
    sha = model_info(repo).sha
    llm = LLM(model=repo, revision=sha, dtype='bfloat16', max_model_len=4096, gpu_memory_utilization=0.85, seed=0)
    return llm, {'name': name, 'id': repo, 'revision': sha, 'vllm': vllm.__version__, 'gpu': GPU,
                 'thinking': False, **SAMPLING}

def run(llm, info, exp, split, n=None):
    pairs = requests_for(exp, split, n)
    msgs = [[{'role': 'system', 'content': rq.instructions}, {'role': 'user', 'content': rq.input}] for _, rq in pairs]
    outs = llm.chat(msgs, SamplingParams(**SAMPLING), chat_template_kwargs={'enable_thinking': False}, use_tqdm=True)
    with RunLog(f"phase2_{exp}_{split}_{info['name']}", run_config(exp, split, info, n), seed=0) as log:
        for (item, rq), o in zip(pairs, outs):
            log.write(record(item, rq, o.outputs[0].text, {
                'model_version': f"{info['id']}@{info['revision'][:12]}", 'input_tokens': len(o.prompt_token_ids),
                'output_tokens': len(o.outputs[0].token_ids), 'cost': 0.0, 'cache_hit': False}))
    parsed = sum(json.loads(l).get('pred') is not None for l in log.path.open() if '"response"' in l)
    print(f'{exp}/{split}: {len(pairs)} requests, parsed {parsed} -> {log.path.name}')

## Step 1: dev runs (prompt freezing only)

If Qwen3.5 fails to load or is very slow, set `USE_FALLBACK = True` (Qwen3-1.7B / Qwen3-4B) and record the switch in PROGRESS.md.

In [ ]:
USE_FALLBACK = False
DEV = [('e1', 'dev'), ('e1', 'vtg_dev'), ('e2', 'dev'), ('e3', 'dev'), ('solve', 'dev')]
for name in ['qwen-2b', 'qwen-4b']:
    llm, info = load(name, USE_FALLBACK)
    for exp, split in DEV:
        run(llm, info, exp, split)
    del llm; import gc; gc.collect(); torch.cuda.empty_cache()

## Step 2: full test grid (only after the prompt is frozen)

Check that `git log -1` above shows the commit that froze the prompt (`PROMPT_VERSION` in `core/prompts.py`).

In [ ]:
from core.prompts import PROMPT_VERSION; print('prompt', PROMPT_VERSION)
TEST = [('e1', 'test_seen'), ('e1', 'test_heldout'), ('e1', 'vtg_test'),
        ('e2', 'test_seen'), ('e2', 'test_heldout'),
        ('e3', 'test_seen'), ('e3', 'test_heldout'), ('solve', 'test_seen')]
for name in ['qwen-2b', 'qwen-4b']:
    llm, info = load(name, USE_FALLBACK)
    for exp, split in TEST:
        run(llm, info, exp, split)
    del llm; import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
!cd runs && zip -q ../phase2_runs.zip *_phase2_*qwen*.jsonl && cd .. && ls -la phase2_runs.zip
from google.colab import files; files.download('phase2_runs.zip')